# CrashVLM: GPU runs (Colab)

Runs the steps that need a GPU. Every step writes its results to Google Drive and **can be re-run
after a disconnect**: finished clips are skipped.

**Before you start**
1. Runtime → Change runtime type → **GPU** (L4 recommended; T4 works for steps A–C).
2. Upload the folder `data` from `~/crashvlm_colab_upload/` on your Mac (Finder: ⌘⇧G, paste
   `~/crashvlm_colab_upload`) into **MyDrive/crashvlm/**. It holds exactly 7 files (448 MB).
   Do **not** upload any other `data` folder (e.g. the old one inside the repo).
3. Add your GitHub token as a Colab secret named **GITHUB_TOKEN** (key icon in the left sidebar),
   and enable notebook access.

| Step | What | Time (L4) |
|---|---|---|
| A | SigLIP 2 so400m embeddings | ~5 min |
| B | Qwen3.5-4B zero-shot reports | ~15 min |
| C | Qwen3-VL-8B (4-bit) zero-shot reports | ~20 min |
| D | LoRA fine-tuning + evaluation | ~1-2 h |
| E | Learning curve (optional) | ~2 h |


In [ ]:
# 0. Setup: GPU check, Google Drive, data check, code download, install  (~3 min)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob, os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/crashvlm/data"
os.environ["CRASHVLM_DATA"] = DATA

# --- data check: the 7 files from ~/crashvlm_colab_upload/data on the Mac ---
need = ["annotation/pass_a/images.zip", "annotation/clip_index.json", "interim/selection.parquet",
        "sft/train.jsonl", "sft/val.jsonl", "sft/train_f0.25.jsonl", "sft/train_f0.5.jsonl"]
missing = [f for f in need if not os.path.exists(f"{DATA}/{f}")]
if missing:
    print("MISSING in MyDrive/crashvlm/data:", missing)
    print("What is there now:")
    for p in sorted(glob.glob("/content/drive/MyDrive/crashvlm/**", recursive=True))[:25]:
        print("  ", p.replace("/content/drive/", ""))
    raise SystemExit("Upload the folder ~/crashvlm_colab_upload/data to MyDrive/crashvlm/ and wait "
                     "until the upload has finished, then run this cell again.")
size_mb = os.path.getsize(f"{DATA}/annotation/pass_a/images.zip") / 2**20
assert size_mb > 400, f"images.zip is only {size_mb:.0f} MB: upload not finished or incomplete"
print(f"data OK (images.zip {size_mb:.0f} MB)")

# --- code (private repo): key icon -> secret 'GITHUB_TOKEN', notebook access ON ---
token = userdata.get("GITHUB_TOKEN")
repo = "/content/crashvlm"
if not os.path.exists(repo + "/scripts"):
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/Faheem8585/crashvlm.git", repo],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed (check the token): " + r.stderr.replace(token, "***"))
else:
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
os.chdir(repo)
print("code ready:", os.getcwd())
r = subprocess.run("pip -q install -e '.[ml]' bitsandbytes && pip -q install -U transformers",
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + r.stderr[-3000:])
import importlib, sys
sys.path.insert(0, repo + "/src")
importlib.invalidate_caches()
import crashvlm, torch, transformers, peft  # noqa: F401  (fails loudly if anything is missing)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers",
      transformers.__version__, "| CUDA", torch.cuda.is_available())
print("setup done: continue with step A")


## A. SigLIP 2 so400m frame embeddings

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
!python scripts/extract_embeddings.py

## B. Qwen3.5-4B zero-shot crash reports (46 test clips)

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_zs

In [ ]:
# quick look at one report
import json
line = open(os.environ["CRASHVLM_DATA"] + "/vlm/qwen3.5-4b_zs.jsonl").readline()
r = json.loads(line); print(r["parse_ok"], r["issues"]); print(r["text"][:800])

## C. Qwen3-VL-8B in 4-bit

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3-VL-8B-Instruct vlm.load_in_4bit=true vlm.run_name=qwen3-vl-8b-4bit_zs

## D. LoRA fine-tuning of Qwen3.5-4B, then evaluation on the same test clips

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
!python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_lora

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
!python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.adapter=$CRASHVLM_DATA/lora/qwen3.5-4b_lora/adapter vlm.run_name=qwen3.5-4b_lora

## E. (optional) Learning curve: 25 % and 50 % of the training tests

In [ ]:
import os; os.chdir("/content/crashvlm"); os.environ["CRASHVLM_DATA"] = "/content/drive/MyDrive/crashvlm/data"
for f in ["0.25", "0.5"]:
    !python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B sft.train_file=train_f{f}.jsonl vlm.run_name=qwen3.5-4b_lora_f{f}
    !python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.adapter=$CRASHVLM_DATA/lora/qwen3.5-4b_lora_f{f}/adapter vlm.run_name=qwen3.5-4b_lora_f{f}

## Done
Everything is in **MyDrive/crashvlm/data**: `features/`, `vlm/`, `lora/`. Download `features/`,
`vlm/` and `lora/*/train_log.json` into `~/crashvlm_data/` on the Mac, then run `eval_vlm.py`, `run_m3.py`
and `paper_tables.py`.
